# DuckDB Live Walkthrough

YZV 322E — Tool Presentation. Run cells top-to-bottom during the demo.

Make sure `bash scripts/00_download_data.sh` has been run first.

In [ ]:
import duckdb, time, os
print('DuckDB version:', duckdb.__version__)
print('CWD:', os.getcwd())

## 1. Query a Parquet file with one line of SQL

In [ ]:
duckdb.sql("""
    SELECT COUNT(*) AS rows
    FROM '../data/yellow_2024-*.parquet'
""").show()

## 2. Real analytical query — top pickup zones by revenue

In [ ]:
t0 = time.perf_counter()
df = duckdb.sql("""
    SELECT
        PULocationID,
        COUNT(*)            AS trips,
        ROUND(AVG(fare_amount)::NUMERIC, 2)  AS avg_fare,
        ROUND(SUM(total_amount)::NUMERIC, 2) AS total_revenue
    FROM '../data/yellow_2024-*.parquet'
    GROUP BY PULocationID
    ORDER BY total_revenue DESC
    LIMIT 10
""").df()
print(f'Query time: {time.perf_counter()-t0:.3f} s')
df

## 3. Pandas DataFrame as a SQL table — zero-copy

In [ ]:
import pandas as pd
small = pd.read_parquet('../data/yellow_2024-01.parquet').head(1000)
duckdb.sql("SELECT payment_type, AVG(tip_amount) AS avg_tip FROM small GROUP BY payment_type").show()

## 4. PIVOT — DuckDB SQL extensions Pandas users will love

In [ ]:
duckdb.sql("""
    PIVOT (
        SELECT
            EXTRACT(hour FROM tpep_pickup_datetime) AS hr,
            payment_type,
            COUNT(*) AS trips
        FROM '../data/yellow_2024-01.parquet'
        WHERE payment_type IN (1, 2)
        GROUP BY hr, payment_type
    )
    ON payment_type
    USING SUM(trips)
    ORDER BY hr
""").show()